## Step 1: Import the required libraries
Import NumPy and TensorFlow/Keras components used to build and train the CBOW neural network.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 2: Prepare the data
Split the sentence into words, create a sorted vocabulary of unique words, and create mappings between words and numerical IDs.

In [2]:
sentence = "the cat sits on the mat"

words = sentence.split()
vocab = sorted(set(words))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print("Words:", words)
print("Vocabulary:", word_to_id)

Words: ['the', 'cat', 'sits', 'on', 'the', 'mat']
Vocabulary: {'cat': 0, 'mat': 1, 'on': 2, 'sits': 3, 'the': 4}


**Explanation**
- `split()` separates the sentence into words.
- `set()` keeps only unique words.
- `word_to_id` assigns a numerical ID to each word.
- `id_to_word` converts IDs back into words.

## Step 3: Generate training data
Use a window size of 1. The words immediately before and after each target word become the context; the middle word is the target.

In [3]:
X = []
y = []

for i in range(1, len(words) - 1):
    context = [
        word_to_id[words[i - 1]],
        word_to_id[words[i + 1]]
    ]

    target = word_to_id[words[i]]

    X.append(context)
    y.append(target)

X = np.array(X)
y = np.array(y)

print("Training contexts (X):")
print(X)
print("Target word IDs (y):")
print(y)
print("Example context words:", [id_to_word[i] for i in X[1]])
print("Example target word:", id_to_word[y[1]])

Training contexts (X):
[[4 3]
 [0 2]
 [3 4]
 [2 1]]
Target word IDs (y):
[0 3 2 4]
Example context words: ['cat', 'on']
Example target word: sits


**Explanation**
- `X` contains the IDs of the surrounding context words.
- `y` contains the ID of the target word.
- For the example context `['cat', 'on']`, the target word is `sits`.

## Step 4: Build and train the CBOW model
The embedding layer converts word IDs into vectors. Global average pooling averages the context vectors, and the Dense softmax layer predicts a probability for every vocabulary word.

In [4]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Embedding(len(vocab), 10),
    layers.GlobalAveragePooling1D(),
    layers.Dense(len(vocab), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 2, 10)          │            50 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 10)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │            55 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105 (420.00 B)

 Trainable params: 105 (420.00 B)

 Non-trainable params: 0 (0.00 B)

### Train the model
Train on the generated context-target examples for 200 epochs, as specified in the practical.

In [5]:
history = model.fit(X, y, epochs=200, verbose=0)

print("Training completed.")
print("Final training accuracy:", round(float(history.history["accuracy"][-1]), 3))
print("Final training loss:", round(float(history.history["loss"][-1]), 4))

Training completed.
Final training accuracy: 0.75
Final training loss: 1.0654


**Explanation**
- **Embedding:** Converts word IDs into numerical vectors.
- **GlobalAveragePooling1D:** Averages the context word vectors.
- **Dense:** Predicts the target word.
- **Softmax:** Produces probabilities for all words.
- `fit()` trains the model using the generated examples.

## Step 5: Predict the target word
Provide the context words `cat` and `on`, then select the vocabulary word with the highest predicted probability.

In [6]:
context_words = ["cat", "on"]

context_ids = np.array([[
    word_to_id[context_words[0]],
    word_to_id[context_words[1]]
]])

prediction = model.predict(context_ids, verbose=0)

predicted_id = np.argmax(prediction)
predicted_word = id_to_word[predicted_id]

print("Context words:", context_words)
print("Predicted word:", predicted_word)
print("Probability:", round(float(prediction[0][predicted_id]), 3))

Context words: ['cat', 'on']
Predicted word: sits
Probability: 0.425


## Result

The Continuous Bag of Words (CBOW) model was implemented using TensorFlow/Keras. It was trained to predict a target word from its surrounding context words. For the context **“cat”** and **“on”**, the expected prediction is **“sits”** after successful training on this small example. The exact probability may vary.

---

## Viva-Voce Questions and Answers

1. **What is CBOW?**  
   CBOW (Continuous Bag of Words) is a Word2Vec model that predicts a target word using its surrounding context words.

2. **What is Word2Vec?**  
   Word2Vec is a group of neural-network-based methods used to learn vector representations of words.

3. **What is the main objective of CBOW?**  
   To predict a missing or target word from the surrounding words.

4. **What is NLP?**  
   Natural Language Processing is a field of AI that enables computers to process and understand human language.

5. **What is a context word?**  
   A context word is a word surrounding the target word that is used as input to the model.

6. **What is a target word?**  
   The target word is the word the model is trained to predict.

7. **What is a window size?**  
   It determines how many nearby words are considered as context. Here, a window size of 1 uses the immediately preceding and following words.

8. **What does the sentence `the cat sits on the mat` produce as an example?**  
   For target `sits`, the context words are `cat` and `on`.

9. **Why do we tokenize the sentence?**  
   To split text into individual tokens that can be processed by the model.

10. **Why is a vocabulary created?**  
    It contains the unique words that the model can represent and predict.

11. **Why convert words into numerical IDs?**  
    Neural networks operate on numerical inputs, so words are mapped to integer IDs.

12. **What is `word_to_id`?**  
    A dictionary that maps each vocabulary word to its integer ID.

13. **What is `id_to_word`?**  
    A dictionary that maps integer IDs back to their corresponding words.

14. **What are `X` and `y` in this practical?**  
    `X` contains context-word IDs, while `y` contains target-word IDs.

15. **What is an embedding?**  
    An embedding is a dense numerical vector representing a word.

16. **What does the Embedding layer do?**  
    It converts integer word IDs into trainable dense vectors.

17. **What does `Embedding(len(vocab), 10)` mean?**  
    It creates an embedding table with one row per vocabulary word and 10 values per word vector.

18. **What is GlobalAveragePooling1D?**  
    It averages the context word vectors to produce one combined context representation.

19. **Why is a Dense layer used?**  
    It maps the combined context representation to scores for the vocabulary words.

20. **Why is softmax used?**  
    Softmax converts output scores into a probability distribution over the vocabulary.

21. **What does the output probability represent?**  
    It represents the model's estimated probability for each possible target word.

22. **What is an epoch?**  
    One complete pass through the training dataset.

23. **Why are 200 epochs used here?**  
    The practical specifies 200 training epochs for this tiny example dataset.

24. **What is the Adam optimizer?**  
    Adam is an optimization algorithm that updates model parameters using adaptive learning rates.

25. **What is a loss function?**  
    It measures the difference between the model's predictions and the actual target values.

26. **Why use sparse categorical crossentropy?**  
    It is suitable for multi-class classification when target labels are integer class IDs.

27. **What is accuracy?**  
    Accuracy is the proportion of predictions that correctly match the target labels.

28. **What does `model.fit()` do?**  
    It trains the model using the supplied inputs and target labels.

29. **What does `model.predict()` do?**  
    It generates model predictions for new input data.

30. **What does `np.argmax()` do?**  
    It returns the index of the largest value, used here to select the most probable word.

31. **What is the purpose of `verbose=0`?**  
    It suppresses the training or prediction progress output.

32. **What is the difference between CBOW and Skip-gram?**  
    CBOW predicts a target word from its context; Skip-gram predicts surrounding context words from a target word.

33. **What are common applications of word embeddings?**  
    Text classification, sentiment analysis, information retrieval, machine translation, and semantic similarity.

34. **What is a limitation of this practical's dataset?**  
    It uses only one short sentence, so it is a demonstration and cannot learn robust general-purpose word representations.

35. **Will the exact predicted probability always be the same?**  
    No. It can vary due to random initialization and training behavior, even though the expected predicted word is `sits`.

## Important Parameters

| Parameter | Value | Purpose |
|---|---:|---|
| Sentence | `the cat sits on the mat` | Input text |
| Window size | 1 | Uses immediate left and right context |
| Embedding dimension | 10 | Length of each word vector |
| Optimizer | Adam | Updates model weights |
| Loss | Sparse categorical crossentropy | Measures prediction error |
| Epochs | 200 | Number of training passes |
| Output activation | Softmax | Produces word probabilities |